In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# dbutils.widgets.text("catalog","de_dev")

In [0]:
catalog = dbutils.widgets.get("catalog")
print(catalog)

In [0]:
df = spark.table(f"{catalog}.bronze.customers")
clean_df = (
    df
    .dropna(subset=["customer_id" , "email"])
    .dropDuplicates(["customer_id"])
    .withColumn("name", trim(col("name")))
    .withColumn("email", lower(col("email")))
    .withColumn("city", initcap(col("city")))
    .withColumn("state", upper(col("state")))
    .select(
        "customer_id",
        "name",
        "email",
        "city",
        "state",
        "signup_date",
        "phone"
    )
    )
clean_df.createOrReplaceTempView("customers_clean_view")
# display(clean_df)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.silver.customers_scd_1 (
    customer_id INT,
    name STRING,
    email STRING,
    city STRING,
    state STRING,
    signup_date DATE,
    phone STRING
)
USING DELTA;

In [0]:
%sql
MERGE INTO ${catalog}.silver.customers_scd_1 AS trg
USING customers_clean_view AS src
ON trg.customer_id = src.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *  

In [0]:
%sql

-- select * from ${catalog}.silver.customers_scd_1 limit 1000